# Hematokrit v CVXPY — napiš si to sám

## Zadání

> Hematokrit $H$ je objemový podíl červených krvinek v krvi. Krvinky nesou
> kyslík, takže čím vyšší hematokrit, tím víc kyslíku v každém mililitru krve.
> S hematokritem ale roste viskozita, takže krev teče pomaleji.
>
> **Při jakém hematokritu se do tkání dostane nejvíc kyslíku?**

$$
\begin{aligned}
\text{maximize}_{H}\quad & D(H) = H\,e^{-\alpha H}
  && \text{dodavka kysliku (rel. jednotky)}\\
\text{subject to}\quad & 0{,}20 \le H \le 0{,}60
  && \text{rozsah, kde plati fit viskozity}
\end{aligned}
$$

Data: $\alpha = 2{,}5$.

**Úkol:** zapsat tuhle úlohu v CVXPY, vyřešit ji a porovnat výsledek s tím, co
vyšlo na tabuli. Níž je přehled syntaxe na třech malých příkladech, které
s hematokritem nesouvisí — jsou tam proto, aby se formulace dala **odvodit**,
ne opsat.

In [ ]:
try:
    import cvxpy as cp
except ImportError:
    %pip install -q cvxpy
    import cvxpy as cp

In [ ]:
import cvxpy as cp
import numpy as np

## Co je CVXPY

Knihovna pro **konvexní** optimalizaci. Úloha se v ní nepíše jako algoritmus, ale
skoro jako na papíře — proměnná, účelová funkce, seznam omezení. CVXPY z toho
sám pozná, jestli je úloha konvexní (`is_dcp()`), přeloží ji do standardního
tvaru, pustí na ni numerický solver (CLARABEL, SCS, OSQP…) a vrátí řešení
i jeho stav. **Není to obecný optimalizátor:** co neumí označit za konvexní, to
odmítne.

## Syntaxe na třech příkladech

Každá úloha má v CVXPY tytéž čtyři kusy: **proměnnou**, **účelovou funkci**,
**seznam omezení** a **složení do `cp.Problem`**. Pak už se jen volá `solve()`.

**1) Jedna proměnná a meze.** Nejlepší nepřípustný bod je $x = 3$, jenže ten do
mezí nespadne — optimum proto skončí **na mezi** a ta je pak *aktivní*.

$$
\begin{aligned}
\text{minimize}_{x}\quad & (x-3)^2\\
\text{subject to}\quad & 0 \le x \le 2
\end{aligned}
$$

In [ ]:
# 1) jedna proměnná, meze -- a mez, která je v optimu aktivní
x = cp.Variable()                          # jedno reálné číslo, které solver hledá
ucel = cp.Minimize(cp.square(x - 3))       # co se minimalizuje
omezeni = [x >= 0, x <= 2]                 # seznam podmínek

uloha = cp.Problem(ucel, omezeni)
print("je to konvexní (DCP)?", uloha.is_dcp())
uloha.solve()
print(f"stav: {uloha.status},  x* = {float(x.value):.4f},  účelová funkce = {uloha.value:.4f}")
print("optimum sedí na horní mezi -- ta mez je AKTIVNÍ")

**2) Maximalizace a logaritmus.** Účelová funkce je konkávní, takže se smí
maximalizovat rovnou; `pos=True` říká, že proměnná je kladná, jinak by logaritmus
neměl smysl.

$$
\begin{aligned}
\text{maximize}_{y}\quad & \ln y - y\\
\text{subject to}\quad & 0 < y \le 5
\end{aligned}
$$

In [ ]:
# 2) maximalizace a logaritmus; pos=True říká, že proměnná je kladná
y = cp.Variable(pos=True)
uloha = cp.Problem(cp.Maximize(cp.log(y) - y), [y <= 5])
uloha.solve()
print(f"y* = {float(y.value):.4f}   (maximalizovat f je totéž co minimalizovat -f)")

**3) Vektorová proměnná a rovnice.** Proměnná nemusí být jedno číslo; omezení
může být i rovnost. Hledá se nejbližší bod k $(1,2,3)$ mezi těmi, které mají
nezáporné složky se součtem 3.

$$
\begin{aligned}
\text{minimize}_{\mathbf z \in \mathbb R^3}\quad
  & \lVert \mathbf z - (1,2,3)\rVert_2^2\\
\text{subject to}\quad & \textstyle\sum_i z_i = 3,\qquad \mathbf z \ge 0
\end{aligned}
$$

In [ ]:
# 3) víc proměnných najednou: proměnná může být vektor a omezení rovnice
z = cp.Variable(3)
uloha = cp.Problem(cp.Minimize(cp.sum_squares(z - np.array([1.0, 2.0, 3.0]))),
                   [cp.sum(z) == 3, z >= 0])
uloha.solve()
print("z* =", np.round(z.value, 4), "  součet =", round(float(np.sum(z.value)), 4))

### Přehled

| co potřebuju | jak se to píše |
|---|---|
| proměnná | `cp.Variable()`, `cp.Variable(pos=True)`, `cp.Variable(n)` |
| účelová funkce | `cp.Minimize(vyraz)` nebo `cp.Maximize(vyraz)` |
| omezení | seznam: `[x >= 0, x <= 1, cp.sum(x) == 1]` |
| úloha | `uloha = cp.Problem(ucel, omezeni)` |
| je to konvexní? | `uloha.is_dcp()` |
| vyřešit | `uloha.solve()` |
| výsledky | `x.value`, `uloha.value`, `uloha.status` |
| užitečné funkce | `cp.square`, `cp.sqrt`, `cp.log`, `cp.exp`, `cp.norm`, `cp.sum_squares` |

Když CVXPY zápis odmítne s `DCPError`, **není to porucha**: je to táž věta jako
na tabuli — v tomhle tvaru se úloha řešit nedá a musí se přepsat.

## Pravidla skládání: podle čeho CVXPY pozná konvexní zápis

CVXPY nederivuje. Skládá výraz z dílů, o kterých ví, jestli jsou konvexní, nebo
konkávní — a podle několika pravidel z toho odvodí, co je celek. Proto některé
zápisy odmítne, ačkoli by je NumPy spolkl.

| pravidlo | příklad zápisu |
|---|---|
| součet konvexních je konvexní | `cp.square(x) + cp.abs(x)` |
| kladný násobek konvexní je konvexní | `3 * cp.square(x)` |
| maximum z konvexních je konvexní | `cp.maximum(x, cp.square(x))` |
| konvexní složená s **afinní** je konvexní | `cp.square(2 * x - 1)` |
| lineární je konvexní **i** konkávní | `2 * x + 1` |
| normy jsou konvexní | `cp.norm(x)` |
| mínus konkávní je konvexní | `-cp.log(x)` |
| ✗ součin dvou výrazů s proměnnou | `x * y`, `x * cp.exp(x)` |
| ✗ minimum ze dvou konvexních | `cp.minimum(cp.square(x), cp.abs(x))` |

**Na hematokritu:** účelová funkce v tom tvaru, do kterého se převedla na tabuli,
je $\alpha H - \ln H$, tedy **lineární člen plus $-\ln H$**. Lineární je konvexní,
$\ln$ je konkávní a mínus konkávní je konvexní — a součet dvou konvexních je
konvexní. Proto tenhle zápis projde. Původní tvar $H\,e^{-\alpha H}$ je naproti
tomu **součin dvou výrazů s proměnnou** a žádné pravidlo na něj nesedí.

Každý výraz se v CVXPY dá zeptat sám:

In [ ]:
t = cp.Variable(pos=True)
vyrazy = (("2.5 * t          (lineární)", 2.5 * t),
          ("-cp.log(t)       (mínus konkávní)", -cp.log(t)),
          ("2.5*t - cp.log(t)  (součet obou)", 2.5 * t - cp.log(t)),
          ("t * cp.exp(-t)   (součin s proměnnou)", t * cp.exp(-t)))
for popis, vyraz in vyrazy:
    print(f"{popis:38s} konvexní? {str(vyraz.is_convex()):5s} konkávní? {vyraz.is_concave()}")

## Úkol: hematokrit

Buňka je schválně prázdná. Postup je v komentářích, kód je na vás.

In [ ]:
ALPHA = 2.5
H_MIN, H_MAX = 0.20, 0.60

# 1) proměnná H
# 2) účelová funkce -- pozor, v jakém tvaru ji CVXPY vezme (viz tabule)
# 3) seznam omezení
# 4) uloha = cp.Problem(...), vypsat uloha.is_dcp() a zavolat uloha.solve()
# 5) vypsat H.value a porovnat s ručním výsledkem 1 / ALPHA

## Až to poběží

1. Sedí $H^\star$ s tím, co vyšlo tužkou? Na kolik desetinných míst?
2. Je některá z mezí aktivní? Co by se stalo, kdyby se z úlohy vyškrtly?
3. Změňte $\alpha$ na 3,0. Kam se optimum posune a proč zrovna tam?
4. Prohoďte schválně `Minimize` a `Maximize`. Solver doběhne a vypíše číslo —
   podle čeho se pozná, že je to nesmysl?